# 01 · Profiling the raw data

**What this notebook is for.** Four decisions in this project were made from
evidence rather than from a document, and this is where that evidence lives. Each
section asks one question of the raw Synthea export, shows the answer, and names
the decision it settled.

Read in order, it is the argument for why the cohort, the measure and the
plausibility range are what they are.

| Question | Settles |
|---|---|
| What arrived, and how much of it? | **D15** — which five of eighteen files to load |
| Which diabetes codes are actually present? | **D5** — who counts as a diabetic patient |
| Is A1c or glucose the right measure? | **D13** — what defines the care gap |
| What range is a plausible A1c? | **DQ3** — the check that was wrong before any defect |

Nothing here writes to the warehouse. It reads the CSVs directly, which is the
point: this ran *before* the pipeline existed.

In [1]:
import sys
sys.path.insert(0, "../src")   # the package lives in src/

import duckdb
import pandas as pd

from caregap.config import RAW, A1C, GLUCOSE, A1C_RANGE, ASOF, sql_list
from caregap.domain.cohort import DIABETES_CODES, EXCLUDED_CODES

pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 62)

con = duckdb.connect()   # in-memory: nothing here touches the warehouse
for t in ("patients", "encounters", "conditions", "observations", "medications"):
    con.sql(f"CREATE VIEW {t} AS SELECT * FROM read_csv_auto('{RAW}/{t}.csv', all_varchar=true)")

def q(sql):
    return con.sql(sql).df()

print("reading from", RAW)

reading from /Users/franklinajisogun/Documents/engineering-project/healthcare-project/clicicalGap/data/raw/csv


## What arrived · Decision D15

Synthea writes eighteen files. The pipeline loads five. The question is which,
and the answer is visible in the row counts.

In [2]:
q(f"""
SELECT 'patients' AS file, count(*) AS rows, true AS loaded FROM patients
UNION ALL SELECT 'encounters',   count(*), true  FROM encounters
UNION ALL SELECT 'conditions',   count(*), true  FROM conditions
UNION ALL SELECT 'observations', count(*), true  FROM observations
UNION ALL SELECT 'medications',  count(*), true  FROM medications
UNION ALL SELECT 'claims_transactions', count(*), false FROM read_csv_auto('{RAW}/claims_transactions.csv', all_varchar=true)
UNION ALL SELECT 'procedures',   count(*), false FROM read_csv_auto('{RAW}/procedures.csv', all_varchar=true)
UNION ALL SELECT 'imaging_studies', count(*), false FROM read_csv_auto('{RAW}/imaging_studies.csv', all_varchar=true)
UNION ALL SELECT 'claims',       count(*), false FROM read_csv_auto('{RAW}/claims.csv', all_varchar=true)
ORDER BY rows DESC
""")

,file,rows,loaded
0,claims_transactions,1094500,False
1,observations,870510,True
2,procedures,187126,False
3,imaging_studies,145496,False
4,claims,127028,False
5,encounters,67755,True
6,medications,59273,True
7,conditions,40811,True
8,patients,1153,True


`claims_transactions` is the largest file in the export and has no clinical role
in an A1c gap — it is a billing ledger. Loading it would make every pipeline run
slower for no page in the app.

**D15: load patients, encounters, conditions, observations, medications.** The
other thirteen are billing, or clinical data with no bearing on this measure.
Scoping it now rather than "load everything and see" is the difference between a
13-second pipeline and a 3-minute one, run dozens of times a week.

## Who counts as diabetic · Decision D5

The obvious approach is to take the type 2 diabetes code and call that the
cohort. Before doing that, look at what else is in the data.

Searching descriptions is how you *discover* codes. It is not how you define a
cohort — that comes next.

In [3]:
q("""
SELECT CODE, DESCRIPTION,
       count(*)                AS n_rows,
       count(DISTINCT PATIENT) AS patients
FROM conditions
WHERE lower(DESCRIPTION) LIKE '%diabet%'
GROUP BY CODE, DESCRIPTION
ORDER BY patients DESC
""")

,CODE,DESCRIPTION,n_rows,patients
0,714628002,Prediabetes (finding),439,439
1,127013003,Disorder of kidney due to diabetes mellitus (disorder),102,102
2,90781000119102,Microalbuminuria due to type 2 diabetes mellitus (disorder),89,89
3,44054006,Diabetes mellitus type 2 (disorder),88,88
4,157141000119108,Proteinuria due to type 2 diabetes mellitus (disorder),64,64
5,368581000119106,Neuropathy due to type 2 diabetes mellitus (disorder),25,25
6,1551000119108,Nonproliferative diabetic retinopathy due to type II diabe...,24,24
7,97331000119101,Macular edema and retinopathy due to type 2 diabetes melli...,3,3
8,1501000119109,Proliferative diabetic retinopathy due to type II diabetes...,1,1


Two things in that table decide the cohort.

**Prediabetes is the largest group, and it is not diabetes.** It is elevated blood
sugar that has not crossed the diagnostic threshold, and those patients do not
qualify for the A1c measure. Include them and the denominator roughly triples, so
every percentage in the finished report is wrong.

**Most of the rest are complications, not diagnoses.** Diabetic kidney disease,
retinopathy, neuropathy. The question is whether the patients carrying them also
carry the type 2 code.

In [4]:
DX = sql_list(DIABETES_CODES)

q(f"""
SELECT (SELECT count(DISTINCT PATIENT) FROM conditions WHERE CODE = '44054006')          AS has_type_2_code,
       (SELECT count(DISTINCT PATIENT) FROM conditions
         WHERE CODE IN ({DX}) AND CODE <> '44054006')                                     AS has_a_complication,
       (SELECT count(DISTINCT PATIENT) FROM conditions
         WHERE CODE IN ({DX}) AND CODE <> '44054006'
           AND PATIENT NOT IN (SELECT PATIENT FROM conditions WHERE CODE = '44054006'))   AS complication_but_no_diagnosis,
       (SELECT count(DISTINCT PATIENT) FROM conditions WHERE CODE IN ({DX}))              AS cohort_all_eight_codes
""")

,has_type_2_code,has_a_complication,complication_but_no_diagnosis,cohort_all_eight_codes
0,88,128,73,161


**That last-but-one number is the finding the whole project turns on.**

Those patients carry a complication *of* diabetes with no diabetes diagnosis code
anywhere on their record. Their chart says, in effect: complication of a disease
nobody wrote down. Anchor the cohort on the single obvious code and they vanish —
and they are the sicker half.

This is not a quirk of synthetic data. It is why HEDIS and every other serious
quality measure is defined by a *value set*, a list of dozens of codes, rather
than one code.

**D5: any of eight SNOMED codes, ever recorded.** Written out with reasons in
`caregap/domain/cohort.py`, including the two deliberate exclusions.

In [5]:
print("In the cohort:")
for code, label in DIABETES_CODES.items():
    print(f"  {code:<16} {label}")
print("\nDeliberately excluded:")
for code, label in EXCLUDED_CODES.items():
    print(f"  {code:<16} {label}")

In the cohort:
  44054006         Diabetes mellitus type 2
  127013003        Disorder of kidney due to diabetes mellitus
  90781000119102   Microalbuminuria due to type 2 diabetes mellitus
  157141000119108  Proteinuria due to type 2 diabetes mellitus
  368581000119106  Neuropathy due to type 2 diabetes mellitus
  1551000119108    Nonproliferative diabetic retinopathy due to type II diabetes mellitus
  97331000119101   Macular edema and retinopathy due to type 2 diabetes mellitus
  1501000119109    Proliferative diabetic retinopathy due to type II diabetes mellitus

Deliberately excluded:
  714628002        Prediabetes (finding)
  80394007         Hyperglycemia (disorder)


One more question decides whether the cohort needs date logic: does a diabetes
diagnosis ever end?

In [6]:
q(f"""
SELECT count(*) AS diagnosis_rows,
       count(STOP) AS rows_with_an_end_date,
       round(100.0 * count(STOP) / count(*), 1) AS pct_resolved
FROM conditions WHERE CODE IN ({DX})
""")

,diagnosis_rows,rows_with_an_end_date,pct_resolved
0,396,0,0.0


Never. So "ever recorded" and "currently active" are the same set here, and the
cohort needs no as-of filter on the diagnosis. Clinically that is right too —
type 2 diabetes is managed, not cured.

Recorded in `cohort.py` as `DIABETES_IS_PERMANENT`, with a note to revisit if that
fraction ever moves.

## A1c or glucose · Decision D13

Glucose is the obvious candidate: it is the bigger table, more patients have one,
and "blood sugar test" is what a layperson would name. All three are reasons to
reject it.

In [7]:
q(f"""
SELECT CODE, DESCRIPTION, UNITS,
       count(*)                AS results,
       count(DISTINCT PATIENT) AS patients,
       min(try_cast(VALUE AS DOUBLE))             AS lo,
       round(median(try_cast(VALUE AS DOUBLE)),1) AS median,
       max(try_cast(VALUE AS DOUBLE))             AS hi
FROM observations
WHERE CODE IN ('{A1C}', '{GLUCOSE[0]}', '{GLUCOSE[1]}')
GROUP BY CODE, DESCRIPTION, UNITS
ORDER BY results DESC
""")

,CODE,DESCRIPTION,UNITS,results,patients,lo,median,hi
0,2339-0,Glucose [Mass/volume] in Blood,mg/dL,10439,532,64.0,85.9,198.3
1,4548-4,Hemoglobin A1c/Hemoglobin.total in Blood,%,8941,497,2.3,4.1,8.8
2,2345-7,Glucose [Mass/volume] in Serum or Plasma,mg/dL,6364,208,64.0,89.1,199.3


### A1c measures control. Glucose measures a moment.

Glucose swings within the hour — a meal takes it from 90 to 140 and back. A1c
measures the fraction of hemoglobin that has become glycated, and since red cells
live about 120 days it integrates roughly three months of exposure.

The difference is measurable in this data. For the same patient, in the same
year:

In [8]:
q(f"""
WITH v AS (
  SELECT CODE, PATIENT, substr(DATE, 1, 4) AS yr,
         max(try_cast(VALUE AS DOUBLE)) - min(try_cast(VALUE AS DOUBLE)) AS spread,
         avg(try_cast(VALUE AS DOUBLE)) AS mean
  FROM observations
  WHERE CODE IN ('{A1C}', '{GLUCOSE[0]}')
  GROUP BY 1, 2, 3 HAVING count(*) > 1
)
SELECT CODE,
       count(*)                                      AS patient_years,
       round(avg(spread), 2)                         AS avg_spread,
       round(avg(100.0 * spread / nullif(mean, 0)),1) AS pct_of_mean
FROM v GROUP BY CODE
""")

,CODE,patient_years,avg_spread,pct_of_mean
0,4548-4,882,0.26,6.7
1,2339-0,1427,18.30,21.3


Glucose is over three times noisier within a single patient. A controlled /
uncontrolled flag built on it would largely detect what time of day someone
happened to visit.

### Glucose is drawn on everyone, so its absence carries no signal

In [9]:
q(f"""
WITH cohort AS (SELECT DISTINCT PATIENT FROM conditions WHERE CODE IN ({DX}))
SELECT CODE,
       count(DISTINCT PATIENT)                                                    AS tested,
       count(DISTINCT PATIENT) FILTER (WHERE PATIENT IN (SELECT PATIENT FROM cohort))     AS diabetic,
       count(DISTINCT PATIENT) FILTER (WHERE PATIENT NOT IN (SELECT PATIENT FROM cohort)) AS not_diabetic
FROM observations
WHERE CODE IN ('{A1C}', '{GLUCOSE[0]}', '{GLUCOSE[1]}')
GROUP BY CODE ORDER BY tested DESC
""")

,CODE,tested,diabetic,not_diabetic
0,2339-0,532,145,387
1,4548-4,497,133,364
2,2345-7,208,115,93


Most people with a glucose result are not diabetic — it rides along on routine
metabolic panels. A gap report built on glucose would find almost nobody, because
almost everybody gets one incidentally.

The measure has to be a test that is ordered *because* the patient has diabetes.
Otherwise a "gap" is just a record of who avoided the phlebotomist.

**D13: HbA1c, LOINC 4548-4.** Glucose stays in the warehouse as patient context
and is never a gap definition. There is also no quality measure anywhere
requiring an annual glucose — HEDIS specifies A1c — so using glucose would discard
the claim that this implements a real published measure.

## What is a plausible A1c · Check DQ3

The data quality spec was written before anyone looked at the data. It proposed
rejecting any A1c outside **3.0 – 20.0 %**. The units are right. The floor is not.

In [10]:
q(f"""
SELECT CASE WHEN v IS NULL  THEN 'not numeric'
            WHEN v < 2.0    THEN 'below 2.0'
            WHEN v < 3.0    THEN 'between 2.0 and 3.0'
            WHEN v > 20.0   THEN 'above 20.0'
            ELSE 'inside the proposed range' END AS bucket,
       count(*) AS results,
       round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
FROM (SELECT try_cast(VALUE AS DOUBLE) AS v FROM observations WHERE CODE = '{A1C}')
GROUP BY 1 ORDER BY results DESC
""")

,bucket,results,pct
0,inside the proposed range,7990,89.4
1,between 2.0 and 3.0,951,10.6


Eleven percent of completely untouched data would fail that check, before a single
defect has been injected.

**Why that matters more than it looks.** The whole project measures a *catch rate*:
249 defects are injected deliberately, logged, and the checks scored against that
known list. A check that fires on clean input makes the number meaningless — it
would be "catching" things that were never broken.

**DQ3 floor revised to 2.0**, which could only be justified by having looked.

The lesson is not that one threshold was wrong. It is that a threshold written
from domain knowledge and never checked against the data is a guess wearing a lab
coat. Profile first, then set the bound.

In [11]:
lo, hi = A1C_RANGE
q(f"""
SELECT {lo} AS revised_floor, {hi} AS ceiling,
       count(*) FILTER (WHERE v < {lo})  AS rejected_now,
       count(*) FILTER (WHERE v < 3.0)   AS would_have_been_rejected,
       count(*) FILTER (WHERE v > {hi})  AS above_ceiling
FROM (SELECT try_cast(VALUE AS DOUBLE) AS v FROM observations WHERE CODE = '{A1C}')
""")

,revised_floor,ceiling,rejected_now,would_have_been_rejected,above_ceiling
0,2.0,20.0,0,951,0


Nothing in this extract exceeds the ceiling, so the upper bound is untested by
real data — which also means the injected value of 250 is a very easy catch, and
the write-up says so rather than implying otherwise.

## What this notebook settled

| Decision | Evidence above |
|---|---|
| **D15** — five files, not eighteen | `claims_transactions` is the largest file and has no clinical role |
| **D5** — eight codes, not one | patients carry a complication with no diagnosis code |
| **D5** — no as-of filter on diagnosis | no diabetes diagnosis in this data ever ends |
| **D13** — A1c, not glucose | glucose is 3× noisier within a patient, and drawn on everyone |
| **DQ3** — floor 2.0, not 3.0 | the proposed floor rejects 11% of clean data |

Each is written up with its reasoning in `docs/DECISIONS.md`, and each is asserted
in `tests/` so a later change has to be deliberate.

Next: `02_validate.ipynb` builds the checks these decisions imply.

In [12]:
con.close()
print("connection closed")

connection closed
